# CDK20のドッキングのためのポケット探索


## AlphaFold DB predicted structure for CDK20 (UniProt entry name lookup)

CDK20 (cyclin-dependent kinase 20, UniProt `Q8IZL9`) has no experimental
structure in RCSB PDB at all -- `chem.rcsb.download_structures("CDK20_HUMAN")`
raises `ValueError: no PDB entries found for UniProt accession Q8IZL9`. So
unlike the thrombin notebook, there is no bound ligand and no crystal
structure to anchor `chem.protein.find_pocket` on anywhere in this notebook
-- pocket detection here relies entirely on a blind scan
(`chem.protein.list_pockets`) of the AlphaFold model, and the resulting
pocket is meant to seed a docking box, not to be cross-checked against known
binders.

`chem.alphafold.download_structures` resolves the id to a UniProt accession
and downloads every AlphaFold DB prediction entry for it.

In [ ]:
from chem import alphafold

n = alphafold.download_structures(
    "CDK20_HUMAN",
    outdir="cdk20_af_data",
    filetype="pdb",
)
n

### Selecting the canonical isoform

UniProt `Q8IZL9` has several annotated splice isoforms, each predicted
separately by AlphaFold DB as its own entry (`AF-Q8IZL9-2-F1`,
`AF-Q8IZL9-3-F1`, ... -- shorter, lower-confidence models), alongside the
canonical full-length entry `AF-Q8IZL9-F1` (no isoform number). Naively
sorting the downloaded filenames and taking the first one would silently
pick an isoform instead of the canonical entry (`"-2-"` sorts before
`"-F1"` as plain strings) -- so the canonical file is selected explicitly by
its AlphaFold entry id shape (`AF-<accession>-F<fragment>.pdb`, exactly one
identifier segment between `AF` and the fragment number).

In [ ]:
import os
import re

from chem import view3d

pdb_files = sorted(f for f in os.listdir("cdk20_af_data") if f.endswith(".pdb"))
print(pdb_files)

_canonical_re = re.compile(r"^AF-[^-]+-F\d+\.pdb$")
canonical_file = next(f for f in pdb_files if _canonical_re.match(f))
print("canonical entry:", canonical_file)

# AlphaFold stores per-residue pLDDT confidence in the B-factor column.
view3d.render_protein(
    os.path.join("cdk20_af_data", canonical_file), coloring="bfactor", width=600, height=600
)

## Blind pocket detection with chem.protein.list_pockets

No bound ligand and no experimental structure means `chem.protein.find_pocket`
(which picks the pocket nearest a *given* ligand) doesn't apply here.
`chem.protein.list_pockets` runs fpocket over the whole canonical structure
and returns every candidate pocket it detects, sorted by druggability score
descending -- exactly the blind-detection case it exists for.

In [ ]:
import pandas as pd

from chem import protein

pockets = protein.list_pockets(os.path.join("cdk20_af_data", canonical_file))
pockets_df = pd.DataFrame(
    [
        {
            "pocket_id": p["pocket_id"],
            "score": p["score"],
            "druggability_score": p["druggability_score"],
            "volume": p["volume"],
            "n_residues": len(p["residues"]),
        }
        for p in pockets
    ]
)
print(f"{len(pockets_df)} candidate pocket(s) found")
display(pockets_df.style.hide(axis="index").format(precision=3))

### Visualize the top pocket

Highlight the top-ranked (highest `druggability_score`) pocket's lining
residues on the structure, on top of a translucent cartoon so they stand
out, alongside its alpha spheres (`spheres` -- fpocket's own Voronoi
vertices approximating the cavity's shape/volume, semi-transparent) so the
cavity itself is visible, not just the residues lining it.

In [ ]:
import py3Dmol
from IPython.display import HTML, display

with open(os.path.join("cdk20_af_data", canonical_file)) as f:
    pdb_text = f.read()

top_pocket = pockets[0]  # list_pockets sorts by druggability_score descending
top_resnums = sorted({r["resnum"] for r in top_pocket["residues"]})

view = py3Dmol.view(width=600, height=450)
view.addModel(pdb_text, "pdb")
view.setStyle({"cartoon": {"color": "lightgrey", "opacity": 0.6}})
view.addStyle({"resi": top_resnums}, {"stick": {"colorscheme": "orangeCarbon"}})
for sphere in top_pocket["spheres"]:
    view.addSphere(
        {
            "center": {"x": sphere["x"], "y": sphere["y"], "z": sphere["z"]},
            "radius": sphere["radius"],
            "color": "cyan",
            "opacity": 0.5,
        }
    )
view.zoomTo({"resi": top_resnums})
view.show()

## Docking box from the top pocket's alpha spheres

`chem.protein.list_pockets`'s `spheres` field is fpocket's own alpha-sphere
vertices approximating the pocket cavity's shape/volume. There's no
ready-made "docking box" helper in `chem.protein` yet, so the box is built
directly here: an axis-aligned bounding box around every alpha sphere
(sphere center ± its own radius, so the box fully contains the cavity, not
just the sphere centers), then a fixed `padding` margin added on every side
(default 4Å, room for the ligand's own conformational/positional freedom
during docking). The result (`center_x/y/z` + `size_x/y/z`) is exactly the
shape AutoDock Vina's `--center_x/y/z`/`--size_x/y/z` flags (or an
equivalent `config.txt`) expect.

In [ ]:
import numpy as np


def docking_box(pocket, padding=4.0):
    """Axis-aligned (center, size) docking box around a list_pockets/find_pocket
    result's alpha spheres, in Angstroms -- ready for AutoDock Vina's
    center_x/y/z + size_x/y/z. `padding` is added on every side beyond the
    spheres' own extent, to leave room for the ligand's conformational/
    positional freedom during docking.
    """
    spheres = pocket["spheres"]
    if not spheres:
        raise ValueError("pocket has no alpha spheres to build a box from")
    centers = np.array([[s["x"], s["y"], s["z"]] for s in spheres])
    radii = np.array([s["radius"] for s in spheres])
    mins = (centers - radii[:, None]).min(axis=0)
    maxs = (centers + radii[:, None]).max(axis=0)
    center = (mins + maxs) / 2
    size = (maxs - mins) + 2 * padding
    return {
        "center_x": round(float(center[0]), 2),
        "center_y": round(float(center[1]), 2),
        "center_z": round(float(center[2]), 2),
        "size_x": round(float(size[0]), 2),
        "size_y": round(float(size[1]), 2),
        "size_z": round(float(size[2]), 2),
    }


box = docking_box(top_pocket)
print(f"pocket {top_pocket['pocket_id']} (druggability_score={top_pocket['druggability_score']}), padding=4.0Å:")
for k, v in box.items():
    print(f"  {k:9s} = {v}")

### Visualize the docking box alongside the pocket

Same view as above (residues + alpha spheres), with the computed box drawn
as a wireframe cube around them, to sanity-check that it actually contains
the cavity before handing the numbers to a docking tool.

In [ ]:
view = py3Dmol.view(width=600, height=450)
view.addModel(pdb_text, "pdb")
view.setStyle({"cartoon": {"color": "lightgrey", "opacity": 0.6}})
view.addStyle({"resi": top_resnums}, {"stick": {"colorscheme": "orangeCarbon"}})
for sphere in top_pocket["spheres"]:
    view.addSphere(
        {
            "center": {"x": sphere["x"], "y": sphere["y"], "z": sphere["z"]},
            "radius": sphere["radius"],
            "color": "cyan",
            "opacity": 0.5,
        }
    )
view.addBox(
    {
        "center": {"x": box["center_x"], "y": box["center_y"], "z": box["center_z"]},
        "dimensions": {"w": box["size_x"], "h": box["size_y"], "d": box["size_z"]},
        "color": "skyblue",
        "wireframe": True,
        "linewidth": 2,
    }
)
view.zoomTo({"resi": top_resnums})
view.show()